**Using BERT to classify sentences**  
https://jalammar.github.io/a-visual-guide-to-using-bert-for-the-first-time/

**BERT Paragraph Vector Embeding w/ Transformers (PyTorch, Colab)**  
https://www.youtube.com/watch?v=wvk5uxMwMYs&t=445s

Nous allons vectoriser les commentaires issus de Reddit en utilisant le code fourni dans ces deux tutoriels.

In [1]:
import os
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, TensorDataset

import nltk
from nltk import sent_tokenize
from transformers import DistilBertTokenizer, DistilBertModel
from transformers import BertTokenizer, BertModel

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import train_test_split

In [2]:
# Lecture des jeux de données
train = pd.read_excel('../data/training_datasets/train_dataset_40pc.xlsx')
test = pd.read_excel('../data/test_dataset_10.xlsx')

train['text_post'] = train['text_post'].astype(str).sample(20000)
test['text_post'] = test['text_post'].astype(str).sample(10000)

X_train, y_train = train.text_post, train.category
X_test, y_test = test.text_post, test.category

In [5]:
chunk_size = 1000  # Smaller chunk size to fit into memory
batch_size = 64
output_dir = "hidden_states_chunks"  # Directory to save intermediate results
os.makedirs(output_dir, exist_ok=True)

# Initialize tokenizer and model
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")
model = DistilBertModel.from_pretrained("distilbert-base-uncased")

# Process dataset in chunks
for chunk_num, i in enumerate(range(0, len(X_train), chunk_size), start=1):
    subset = X_train[i:i + chunk_size].astype(str).tolist()

    # Tokenize lazily
    encoded_dict = [
        tokenizer(
            post,
            add_special_tokens=True,
            max_length=256,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        ) for post in subset
    ]
    input_ids = torch.cat([e["input_ids"] for e in encoded_dict], dim=0)
    attention_masks = torch.cat([e["attention_mask"] for e in encoded_dict], dim=0)

    # Create DataLoader
    dataset = TensorDataset(input_ids, attention_masks)
    dataloader = DataLoader(dataset, batch_size=batch_size)

    # Process each batch
    chunk_hidden_states = []
    with torch.no_grad():
        for batch in dataloader:
            batch_input_ids, batch_attention_masks = [b for b in batch]
            outputs = model(batch_input_ids, attention_mask=batch_attention_masks)
            chunk_hidden_states.append(outputs.last_hidden_state.cpu())  # Move to CPU to free GPU memory

    # Save results for this chunk
    chunk_hidden_states = torch.cat(chunk_hidden_states, dim=0)
    torch.save(chunk_hidden_states, os.path.join(output_dir, f"chunk_{chunk_num}.pt"))
    print(f"Chunk {chunk_num} processed and saved.")

Chunk 1 processed and saved.
Chunk 2 processed and saved.
Chunk 3 processed and saved.
Chunk 4 processed and saved.
Chunk 5 processed and saved.
Chunk 6 processed and saved.
Chunk 7 processed and saved.
Chunk 8 processed and saved.
Chunk 9 processed and saved.
Chunk 10 processed and saved.
Chunk 11 processed and saved.
Chunk 12 processed and saved.
Chunk 13 processed and saved.
Chunk 14 processed and saved.
Chunk 15 processed and saved.
Chunk 16 processed and saved.
Chunk 17 processed and saved.
Chunk 18 processed and saved.
Chunk 19 processed and saved.
Chunk 20 processed and saved.
Chunk 21 processed and saved.
Chunk 22 processed and saved.
Chunk 23 processed and saved.
Chunk 24 processed and saved.
Chunk 25 processed and saved.
Chunk 26 processed and saved.
Chunk 27 processed and saved.
Chunk 28 processed and saved.
Chunk 29 processed and saved.
Chunk 30 processed and saved.
Chunk 31 processed and saved.
Chunk 32 processed and saved.
Chunk 33 processed and saved.
Chunk 34 processed 

In [ ]:
# List to store all chunks' hidden states
all_hidden_states = []

# Iterate over the saved chunk files and load each one
for chunk_num in range(1, len(os.listdir(output_dir)) + 1):
    chunk_file = os.path.join(output_dir, f"chunk_{chunk_num}.pt")

    if os.path.exists(chunk_file):
        chunk_hidden_states = torch.load(chunk_file)
        all_hidden_states.append(chunk_hidden_states)
        print(f"Chunk {chunk_num} loaded.")
    else:
        print(f"Chunk {chunk_num} file not found!")

# Concatenate all the loaded chunk hidden states into a single tensor
all_hidden_states = torch.cat(all_hidden_states, dim=0)
print(f"All chunks combined: {all_hidden_states.shape}")


In [ ]:
 # Slice the output for the first position for all the sequences, take all hidden unit outputs
X = cls_hidden_states = chunk_hidden_states[:, 0, :]  # Shape (num_posts, hidden_size)
y = y_train

In [ ]:
clf = LogisticRegression(max_iter=150)
clf.fit(X, y)


In [ ]:
clf.score(X_test, y_test)